# Querying notebook


Create a TextSearchIndex object, and use the SQLite database file called faq.db.

If faq.db already exists, it opens/connects to it.

If faq.db does not exist, it creates it.

Important: "faq.db" is a relative path, so Python looks for it in the current project folder.

In [3]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [4]:
sqlite_index.count()

85

In [5]:
sqlite_index

It takes your query:

Can I still join the course after it started?

Looks inside the indexed text fields (inside all of them):

["question", "section", "answer"]

Finds documents with similar words like: <br>
join <br>
course <br>
started <br>
late <br>

Ranks them by relevance, then returns the best matches. <br>
When it finds a match, it returns the whole document dictionary, not only the matching field.

In [6]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)

In [7]:
# results is a list of dictionaries 
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '0ae5c221b9',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
  'answer': 'To get started you need three things:\n\n1. A Gemini API key saved in your `.env` file, for example as `GEMINI_API_KEY`.\n2. An OpenAI client pointed at Google’s OpenAI-compatible base URL.\n3. Your selected Google Gemini model name in your request.\n\nExample code (loads the API key from `.env`, creates the Gemini client, and defines the `llm` helper):\n\n```python\nimport os\nfrom dotenv import load_dotenv\nfrom openai import OpenAI\n\nload_dotenv()\n\nclient = OpenAI(\n    ap

In [8]:
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'I missed the first homework - can I still get a certificate?',
 'Homework: Why does the content keep changing?',
 'What happens to code saved in Codespaces if I do not commit it?']

# RAG with sqlitesearch

The RAG system does not care where the index comes from,
as long as the index has a .search() method.

And we are  showing that the RAG code is modular.


In [9]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv

openai_client = OpenAI()
load_dotenv()

# creates an object from the class RAGBase
assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [ ]:
# the rag method from the RAGBase class wires it all together
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join after it started. If you want a certificate, you need to submit your project while submissions are still being accepted.


The answer should be similar to what we got with minsearch. But now the data comes from a persistent index - no fetching, no processing, no indexing at startup. And we didn't have to rewrite any of the RAG logic - just swapped the index.

Pick the right tool for your data: <br>
* minsearch: single process, in-memory only, re-indexes on every startup. Use when data is small and indexing is fast.
* sqlitesearch: separate ingestion and query, file-based (SQLite), opens existing index. Use when data is large or ingestion is slow.

For larger production systems, use the same pattern with a different backend:  <br>
Elasticsearch <br>
OpenSearch <br>
Qdrant (vector database) <br>
Weaviate (vector database) <br>

In [12]:
# When you're done, close the database connection
# Or just let Python clean it up when the notebook kernel shuts down.
sqlite_index.close()